In [13]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [51]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum, count, round
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Tugas6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [52]:
#A. Extract
#mengubah csv menjadi parquet agar mudah diolah
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_transaksi.write.mode("overwrite").parquet("tugas6_transaksi_parquet")

df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)
df_ulasan.write.mode("overwrite").parquet("tugas6_ulasan_parquet")

print("data sudah diubah!")

data sudah diubah!


In [61]:
#csv
print("df_transaksi")
df_transaksi = spark.read.parquet("tugas6_transaksi_parquet")
print("Jumlah baris:", df_transaksi.count())
df_transaksi.printSchema()
#Json
print("df_produk")
df_produk = spark.read.json("tugas6_produk.json")
print("Jumlah baris:", df_produk.count())
df_produk.printSchema()
#csv
print("df_ulasan")
df_ulasan = spark.read.parquet("tugas6_ulasan_parquet")
print("Jumlah baris:", df_ulasan.count())
df_ulasan.printSchema()

df_transaksi
Jumlah baris: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

df_produk
Jumlah baris: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

df_ulasan
Jumlah baris: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [54]:
#B. Transform 1
# Join transaksi dengan ulasan menggunakan LEFT JOIN
df_gabung = df_transaksi.join(df_ulasan, on="order_id", how="left")

# Join hasil sebelumnya dengan data produk menggunakan INNER JOIN
df_gabung = df_gabung.join(df_produk, on="product_id", how="inner")

print("Jumlah data setelah join:", df_gabung.count())

df_gabung.show(10)



Jumlah data setelah join: 5000
+----------+--------+------------+-------------------+------+------+------------+-----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|
+----------+--------+------------+-------------------+------+------+------------+-----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|
|        26|     TX6|           6|2026-10-24 00:00:00|     4| 75000|   Kesehatan|  Produk-26|
|         4|     TX7|        

In [55]:
#menambahkan kolom total pendapatan
df_gabung = df_gabung.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga")
)

df_gabung.select(
    "order_id",
    "product_id",
    "nama_produk",
    "unit_terjual",
    "harga",
    "total_pendapatan"
).show(10)

+--------+----------+-----------+------------+------+----------------+
|order_id|product_id|nama_produk|unit_terjual| harga|total_pendapatan|
+--------+----------+-----------+------------+------+----------------+
|     TX0|        21|  Produk-21|           2| 25000|           50000|
|     TX1|         8|   Produk-8|           6|250000|         1500000|
|     TX2|        25|  Produk-25|           4| 25000|          100000|
|     TX3|         3|   Produk-3|           4| 75000|          300000|
|     TX4|        19|  Produk-19|           6| 75000|          450000|
|     TX5|        10|  Produk-10|           3| 75000|          225000|
|     TX6|        26|  Produk-26|           6| 75000|          450000|
|     TX7|         4|   Produk-4|           1|150000|          150000|
|     TX8|         7|   Produk-7|           2| 75000|          150000|
|     TX9|        30|  Produk-30|           1|250000|          250000|
+--------+----------+-----------+------------+------+----------------+
only s

In [56]:
#C. Transform 2 penanganan data kosong
#membuat kolom ulasan
df_gabung = df_gabung.withColumn(
    "ada_ulasan",
    col("rating").isNotNull()
)

df_gabung.select(
    "order_id",
    "rating",
    "ada_ulasan"
).show(10)

#mengisi data kosong null menjadi 0
df_gabung = df_gabung.fillna(
    {"rating": 0}
)

df_gabung.select(
    "order_id",
    "rating",
    "ada_ulasan"
).show(10)

+--------+------+----------+
|order_id|rating|ada_ulasan|
+--------+------+----------+
|     TX0|     1|      true|
|     TX1|  NULL|     false|
|     TX2|     2|      true|
|     TX3|     5|      true|
|     TX4|  NULL|     false|
|     TX5|     5|      true|
|     TX6|     4|      true|
|     TX7|  NULL|     false|
|     TX8|     5|      true|
|     TX9|  NULL|     false|
+--------+------+----------+
only showing top 10 rows

+--------+------+----------+
|order_id|rating|ada_ulasan|
+--------+------+----------+
|     TX0|     1|      true|
|     TX1|     0|     false|
|     TX2|     2|      true|
|     TX3|     5|      true|
|     TX4|     0|     false|
|     TX5|     5|      true|
|     TX6|     4|      true|
|     TX7|     0|     false|
|     TX8|     5|      true|
|     TX9|     0|     false|
+--------+------+----------+
only showing top 10 rows



In [57]:
#cek hasil transformasi data
df_gabung.select(
    "order_id",
    "product_id",
    "nama_produk",
    "kategori",
    "unit_terjual",
    "harga",
    "total_pendapatan",
    "rating",
    "ada_ulasan"
).show(20)

+--------+----------+-----------+------------+------------+------+----------------+------+----------+
|order_id|product_id|nama_produk|    kategori|unit_terjual| harga|total_pendapatan|rating|ada_ulasan|
+--------+----------+-----------+------------+------------+------+----------------+------+----------+
|     TX0|        21|  Produk-21|  Elektronik|           2| 25000|           50000|     1|      true|
|     TX1|         8|   Produk-8|     Fashion|           6|250000|         1500000|     0|     false|
|     TX2|        25|  Produk-25|  Elektronik|           4| 25000|          100000|     2|      true|
|     TX3|         3|   Produk-3|     Fashion|           4| 75000|          300000|     5|      true|
|     TX4|        19|  Produk-19|Rumah Tangga|           6| 75000|          450000|     0|     false|
|     TX5|        10|  Produk-10|  Elektronik|           3| 75000|          225000|     5|      true|
|     TX6|        26|  Produk-26|   Kesehatan|           6| 75000|          450000

In [58]:
#D. Load data ke HDFS
!hdfs dfs -mkdir -p /user/revan18/etl_output

df_gabung.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/revan18/etl_output/tugas6_enriched"
)

print("Pipeline ETL selesai — hasil tersimpan di HDFS.")
!hdfs dfs -ls /user/revan18/etl_output/tugas6_enriched

[Stage 172:>                                                        (0 + 1) / 1]

Pipeline ETL selesai — hasil tersimpan di HDFS.


Found 6 items
-rw-r--r--   3 revan18 supergroup          0 2026-09-30 20:18 /user/revan18/etl_output/tugas6_enriched/_SUCCESS
drwxr-xr-x   - revan18 supergroup          0 2026-09-30 20:18 /user/revan18/etl_output/tugas6_enriched/kategori=Elektronik
drwxr-xr-x   - revan18 supergroup          0 2026-09-30 20:18 /user/revan18/etl_output/tugas6_enriched/kategori=Fashion
drwxr-xr-x   - revan18 supergroup          0 2026-09-30 20:18 /user/revan18/etl_output/tugas6_enriched/kategori=Kesehatan
drwxr-xr-x   - revan18 supergroup          0 2026-09-30 20:18 /user/revan18/etl_output/tugas6_enriched/kategori=Makanan
drwxr-xr-x   - revan18 supergroup          0 2026-09-30 20:18 /user/revan18/etl_output/tugas6_enriched/kategori=Rumah Tangga


In [60]:
#E. Insight akhir
df_insight = df_gabung.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    sum(
        when(col("ada_ulasan") == True, 1).otherwise(0)
    ).alias("transaksi_dengan_ulasan")
)

df_insight = df_insight.withColumn(
    "persentase_ulasan",
    round(
        col("transaksi_dengan_ulasan") /
        col("total_transaksi") * 100,
        2
    )
)

df_insight.orderBy(
    col("persentase_ulasan").asc()
).show()

#ambil kategori terendah
kategori_terendah = df_insight.orderBy(
    col("persentase_ulasan").asc()
).first()

print("Kategori dengan persentase ulasan paling rendah:")
print("Kategori :", kategori_terendah["kategori"])
print("Persentase :", kategori_terendah["persentase_ulasan"], "%")

+------------+---------------+-----------------------+-----------------+
|    kategori|total_transaksi|transaksi_dengan_ulasan|persentase_ulasan|
+------------+---------------+-----------------------+-----------------+
|     Makanan|            536|                    369|            68.84|
|   Kesehatan|            961|                    662|            68.89|
|     Fashion|           1035|                    726|            70.14|
|Rumah Tangga|            815|                    575|            70.55|
|  Elektronik|           1653|                   1168|            70.66|
+------------+---------------+-----------------------+-----------------+

Kategori dengan persentase ulasan paling rendah:
Kategori : Makanan
Persentase : 68.84 %


Berdasarkan hasil ETL, kategori Makanan memiliki persentase transaksi dengan ulasan paling rendah dibandingkan kategori lainnya. Hal ini penting diketahui oleh tim marketing karena jumlah ulasan dapat menjadi salah satu sumber informasi untuk mengetahui pengalaman pelanggan. Tim marketing dapat mendorong pelanggan pada kategori Makanan untuk memberikan ulasan setelah melakukan pembelian.